# H2 · `scripts/qa.py`

## What this file is for

Five tiers, a pairwise matrix, a cost estimate and a budget guard -- the QA programme behind the
`/tests` and QA tab's tier runner. Terminal only; nothing here knows a UI exists.

**Two matrices, not one.** A *value* sweep asks "is every filed rate returned correctly" and needs
no ISO call, because ISO's own files are the answer. A *logic* matrix asks "does the engine behave
correctly" and stays small because the axes key on one another -- both aggregate limits key on the
occurrence limit, so 11,700 naive limit combinations are really 464.

**Pairwise, not exhaustive.** Every measured defect so far needed two things set at once -- a
deductible *and* a limit, size-of-risk *and* a countrywide state. None needed three. `_allpairs`
covers every pair of axis values in a few hundred scenarios where the cross product needs millions.

**Depends on:** [`H1 variants.py`](01-variants.ipynb) for what a configuration means,
[`H4 sweep.py`](04-sweep.py) to run one, [`H5 runstore.py`](05-runstore.ipynb) for the budget's
source of truth.

## Its public surface

Generated from the module, so it can't drift.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))
sys.path.insert(0, str(Path.cwd().parent.parent / "scripts"))

import inspect
import qa

for name, obj in vars(qa).items():
    if name.startswith("_") or getattr(obj, "__module__", None) != qa.__name__:
        continue
    if inspect.isfunction(obj):
        print(f"def {name}{inspect.signature(obj)}")
    elif name.isupper():
        v = repr(obj)
        print(f"{name} = {v if len(v) < 90 else v[:87] + '...'}")

## The smallest thing that works

`--tiers` describes the tiers without running anything; `scenarios_for` and `cost` are the same
thing this prints, callable directly.

In [ ]:
for k in sorted(qa.TIERS):
    t = qa.TIERS[k]
    built = "" if t["build"] else "  NOT BUILT"
    print(f"{k}  {t['name']:<12}{built}")
    print(f"    {t['what']}")

print()
c = qa.cost("T1", jurisdictions=["TX", "CA"], offline=True)
print("T1 in TX and CA, offline:", c)

## The interesting case

### The axes are keyed, so "pairwise" does not mean "independent"

`occurrence_limit` and the deductibles are ordinary axes. But T1 also runs `PAIRED` configurations
-- schedule rating with its percentage, claims-made with its year -- **alongside** the pairwise set
rather than inside it, because pairing a dependent control independently would produce
configurations that are legal and inert: `schedule_pct` without `schedule_rating=Yes` sets a value
ISO ignores.

In [ ]:
sc = qa.scenarios_for("T1", jurisdictions=["TX", "CA"])
print(f"T1 in two states: {len(sc)} scenarios\n")

import variants as V
for cfg, js in sc[:4]:
    print(" ", V.describe(cfg) or "the base risk, unvaried")
print("  ...")
for cfg, js in sc[-3:]:
    print(" ", V.describe(cfg) or "the base risk, unvaried")

### The budget warns; it does not decide

`_budget_check` reads today's actual spend from the run store and states what a tier would cost
against it -- three levels, `OK`, `OVER_STANDING` (still under the 150 ceiling) and
`OVER_CEILING`. It never refuses on its own; `main()` is the caller that stops without `--force`.

In [ ]:
today = qa._spent_today()
print("live calls already spent today:", today)
for n in (5, 48, 120):
    b = qa._budget_check(n)
    print(f"  +{n:<4} -> {b['level']:<14} {b['why']}")

## What it refuses

T3 -- the value sweep, 278,054 filed cells -- is designed and not built. Asking for its scenarios
returns nothing rather than pretending; `main()` checks `spec["build"] is None` and says so before
it would try to run zero scenarios and report success.

In [ ]:
print(qa.TIERS["T3"])
print("scenarios_for('T3'):", qa.scenarios_for("T3"))

## Try it yourself

1. `qa.cost("T2", offline=True)` -- T2 is T1's matrix across all 51 jurisdictions rather than the
   core 12. How many more scenarios buys how many more ratings?
2. Read `_allpairs` -- it is a plain greedy covering algorithm, no dependency. Trace through what
   it does with two two-valued axes by hand; does it produce the 4 pairs you'd expect?
3. `python scripts/qa.py --tier T1 --offline` from a terminal runs what this notebook only plans.
   [`H3 qa_review.py`](03-qa_review.ipynb) is what reads the runs it produces.

In [ ]:
# your turn